In [2]:
import duckdb as db
import pandas as pd
import matplotlib.pyplot as plt

from startorch.utils import PROJECT_ROOT
COMPACT_PATH = "/data/scholar_rank/data/full_corpus"
DB_PATH = "/data/math_english"

In [5]:
con = db.connect()

count = con.sql(f"""
    SELECT field, count(*) AS freq 
    FROM (
        SELECT *, topics[1].field_display_name AS field
        FROM read_parquet('/data/startorch/data/full_corpus/works/**/*.parquet')
        WHERE language = 'en'
    )
    GROUP BY field
    ORDER BY freq DESC
""").fetchall()

con.close()

In [6]:
print(count)

[(None, 78490353), ('Medicine', 43040333), ('Engineering', 33646156), ('Social Sciences', 29771609), ('Computer Science', 18217605), ('Biochemistry, Genetics and Molecular Biology', 16801102), ('Agricultural and Biological Sciences', 16423634), ('Physics and Astronomy', 16354139), ('Arts and Humanities', 13670207), ('Environmental Science', 13370465), ('Economics, Econometrics and Finance', 9113515), ('Materials Science', 8414844), ('Psychology', 6659093), ('Earth and Planetary Sciences', 6307347), ('Business, Management and Accounting', 6030351), ('Chemistry', 5877734), ('Health Professions', 5181086), ('Neuroscience', 3885490), ('Mathematics', 3794495), ('Decision Sciences', 2724199), ('Immunology and Microbiology', 2443848), ('Energy', 1507946), ('Nursing', 1175584), ('Pharmacology, Toxicology and Pharmaceutics', 981753), ('Dentistry', 806638), ('Chemical Engineering', 792447), ('Veterinary', 415820)]


In [5]:
con = db.connect()

count = con.sql(f"""
    SELECT language, count(*) AS freq
    FROM read_parquet('{COMPACT_PATH}/works/**/*.parquet') 
    GROUP BY language
    ORDER BY freq DESC
""").fetchall()

print(count)

con.close()

[('en', 345897793), (None, 28222663), ('de', 23076574), ('es', 18657045), ('fr', 18259365), ('ja', 14428263), ('pt', 9486692), ('zh', 6030004), ('ru', 5498895), ('it', 4917531), ('id', 4357810), ('ko', 3424087), ('nl', 3182540), ('pl', 3110819), ('sv', 1811788), ('tr', 1785240), ('cs', 1744929), ('uk', 1627272), ('ar', 1577309), ('ca', 1286580), ('fi', 1037401), ('hu', 806081), ('no', 789145), ('hr', 670304), ('fa', 608962), ('da', 558286), ('el', 522367), ('sh', 507327), ('sl', 446231), ('ceb', 442064), ('eu', 418320), ('ro', 400636), ('la', 398470), ('lv', 381290), ('th', 353172), ('ka', 290417), ('lt', 288124), ('ms', 273519), ('uz', 238876), ('sr', 212213), ('eo', 195214), ('vi', 172878), ('he', 152929), ('war', 147590), ('et', 143933), ('sk', 140177), ('gl', 106266), ('af', 73833), ('bg', 64990), ('is', 63402), ('hi', 55044), ('br', 54371), ('ga', 51602), ('mk', 49490), ('bs', 46454), ('be', 35564), ('nds', 32770), ('ur', 31718), ('sq', 30213), ('nn', 28553), ('az', 28102), ('an',

In [3]:
con = db.connect()

count = con.sql(f"""
    SELECT count(*) AS freq
    FROM read_parquet('{COMPACT_PATH}/works/**/*.parquet') 
""").fetchall()

print(count)

con.close()

[(510372821,)]


In [ ]:
# Getting maximum works id value
DB_PATH = '/data/math_english'

con = db.connect()

ids = con.sql(f"""
    SELECT regexp_replace(id, 'W', '')::BIGINT AS id,
    FROM read_parquet('{DB_PATH}/**/*.parquet') 
    ORDER BY id DESC
""")
print(DB_PATH)
print(ids.fetchone())
con.close()

In [ ]:
# Index composition stats, computed from the raw doc_id-term mapping table
# (token_stream/*.bin - pre-merge SPIMI input), not from block_meta.bin/
# posting_*.bin. Parsed directly from the real, already-built binary wire
# format in parallel (one process per file), not re-derived from raw
# tokenization - much faster and exactly matches what was actually indexed.
#
# Correctness note: tokenizer.py's fetchmany() batches don't respect
# document boundaries, and a file closes after a fixed batch count
# regardless - so a document's records can legitimately span two adjacent
# files. The real C++ pipeline (construct_doc_len_list.cpp) handles this
# because it processes files sequentially without resetting state at file
# boundaries. Parsing files independently in parallel breaks that
# assumption, so each file's first/last document-run is kept separate from
# its provably-complete "interior" and stitched against its neighbors in a
# single-threaded reduce pass, in file order.
#
# multiprocessing note: Python 3.14 defaults to the "forkserver" start
# method, which needs a real importable __main__ file - breaks under
# Jupyter/IPython. Forcing "fork" (still available on Linux) instead.

import struct
import time
import multiprocessing
import duckdb as db
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

con = db.connect()


def parse_token_file(path_str: str):
    """Parse one token_*.bin file into (interior_df, interior_docs, total_occurrences,
    first_doc_id, first_run_terms, last_doc_id, last_run_terms).

    Every doc-id run strictly inside this file is provably complete and gets
    folded into interior_df immediately; the file's first and last runs are
    kept separate (with their own term sets) so parse_token_stream_dir's
    reduce step can stitch them against neighboring files.
    """
    data = Path(path_str).read_bytes()
    n = len(data)
    pos = 0

    current_doc_id = None
    current_run_terms = None
    interior_df = {}
    interior_docs = 0
    total_occurrences = 0

    first_doc_id = None
    first_run_terms = None

    def close_interior_run(terms):
        nonlocal interior_docs
        for t in terms:
            interior_df[t] = interior_df.get(t, 0) + 1
        interior_docs += 1

    while pos < n:
        doc_id, term_len = struct.unpack_from('<qH', data, pos)
        pos += 10
        term = data[pos:pos + term_len].decode('utf-8')
        pos += term_len
        total_occurrences += 1

        if doc_id != current_doc_id:
            if current_doc_id is not None:
                if first_doc_id is None:
                    first_doc_id = current_doc_id
                    first_run_terms = current_run_terms
                else:
                    close_interior_run(current_run_terms)
            current_doc_id = doc_id
            current_run_terms = set()
        current_run_terms.add(term)

    last_doc_id = current_doc_id
    last_run_terms = current_run_terms

    if first_doc_id is None:
        # pathological: file contains only a single doc-id run total
        first_doc_id = last_doc_id
        first_run_terms = last_run_terms

    return (interior_df, interior_docs, total_occurrences,
            first_doc_id, first_run_terms, last_doc_id, last_run_terms)


def parse_token_stream_dir(dir_path: str, max_workers=None):
    """Parse every token_*.bin file in dir_path in parallel, reduce into a
    single global term -> document-frequency dict.

    Safe to sum each file's *interior* df directly: files cover disjoint,
    contiguous doc_id ranges (tokenizer.py writes ORDER BY id ASC), so no
    doc_id ever appears in two files except possibly split across an
    adjacent pair at exactly the boundary - handled by the stitching pass
    below, which must run over files in their original write order.
    """
    files = sorted(Path(dir_path).glob("*.bin"))  # lexicographic == write order (token_0000.bin, ...)
    file_strs = [str(p) for p in files]
    print(f"Parsing {len(files)} files from {dir_path}...")

    results_by_file = {}
    start = time.monotonic()
    with ProcessPoolExecutor(max_workers=max_workers, mp_context=multiprocessing.get_context("fork")) as pool:
        futures = {pool.submit(parse_token_file, f): f for f in file_strs}
        done = 0
        for future in as_completed(futures):
            f = futures[future]
            results_by_file[f] = future.result()
            done += 1
            elapsed = time.monotonic() - start
            print(f"  [{done}/{len(files)}] {f} done ({elapsed:.1f}s elapsed)")

    print(f"All files parsed in {time.monotonic() - start:.1f}s. Stitching boundaries...")

    global_df = {}
    total_docs = 0
    total_occurrences = 0
    pending_doc_id = None
    pending_terms = None

    def add_doc(terms):
        nonlocal total_docs
        for t in terms:
            global_df[t] = global_df.get(t, 0) + 1
        total_docs += 1

    for f in file_strs:  # must process in the original write order
        interior_df, interior_docs, occ, first_doc_id, first_terms, last_doc_id, last_terms = results_by_file[f]

        total_occurrences += occ
        for term, df in interior_df.items():
            global_df[term] = global_df.get(term, 0) + df
        total_docs += interior_docs

        if pending_doc_id is not None and pending_doc_id == first_doc_id:
            merged = pending_terms | first_terms
        else:
            if pending_doc_id is not None:
                add_doc(pending_terms)
            merged = first_terms
        add_doc(merged)

        pending_doc_id = last_doc_id
        pending_terms = last_terms

    add_doc(pending_terms)  # flush the very last file's trailing run

    elapsed = time.monotonic() - start
    print(f"Finished in {elapsed:.1f}s. {len(global_df):,} unique terms, "
          f"{total_docs:,} docs, {total_occurrences:,} total token occurrences.")
    return global_df, total_occurrences, total_docs


# --- Validate against an independent DuckDB recomputation on the small math_en subset ---
from startorch import Tokenizer

MATH_EN_CORPUS = "/data/scholar_rank/data/works_subset/math_en"
stop_word_list = Tokenizer().stop_word_list

con.sql("INSTALL fts; LOAD fts;")
con.sql(f"""
    CREATE OR REPLACE TEMP TABLE tokens_reference AS
    SELECT
        id,
        unnest(list_transform(
            regexp_extract_all(
                regexp_replace(
                    lower(concat_ws(' ',
                        coalesce(title, ''),
                        list_aggregate(list_transform(topics, t -> t.display_name), 'string_agg', ' '),
                        list_aggregate(list_transform(topics, t -> t.subfield_display_name), 'string_agg', ' '),
                        list_aggregate(list_transform(topics, t -> t.field_display_name), 'string_agg', ' '),
                        list_aggregate(list_transform(topics, t -> t.domain_display_name), 'string_agg', ' '),
                        list_aggregate(list_transform(keywords, k -> k.display_name), 'string_agg', ' ')
                    )),
                    '{stop_word_list}',
                    ' ',
                    'g'
                ),
                '[a-z0-9]+'
            ),
            token -> stem(token, 'english')
        )) AS token
    FROM read_parquet('{MATH_EN_CORPUS}/**/*.parquet')
""")

ref_total_occurrences, ref_total_docs, ref_unique_terms = con.sql("""
    SELECT count(*), count(DISTINCT id), count(DISTINCT token) FROM tokens_reference
""").fetchone()

math_en_df, math_en_occurrences, math_en_docs = parse_token_stream_dir(
    "/data/scholar_rank/posting/math_en/token_stream"
)

print()
print("Validation (binary parser vs. independent DuckDB recomputation, math_en):")
print(f"  total_occurrences: parser={math_en_occurrences:,}  duckdb={ref_total_occurrences:,}  match={math_en_occurrences == ref_total_occurrences}")
print(f"  total_docs:        parser={math_en_docs:,}  duckdb={ref_total_docs:,}  match={math_en_docs == ref_total_docs}")
print(f"  unique_terms:      parser={len(math_en_df):,}  duckdb={ref_unique_terms:,}  match={len(math_en_df) == ref_unique_terms}")

In [ ]:
# The real run: full_en's token_stream is 210GB across 205 files - this is
# the corpus every other benchmark number in bmw_technical_report.md
# describes. Expect roughly 3-5 minutes with 20 cores available (~18s/file,
# ceil(205/20) batches). Cached to disk afterward so re-running this
# notebook doesn't repeat the scan.

import pickle

FULL_EN_TOKEN_STREAM = "/data/scholar_rank/posting/full_en/token_stream"
CACHE_PATH = Path("full_en_term_df_cache.pkl")

if CACHE_PATH.exists():
    print(f"Loading cached results from {CACHE_PATH}...")
    with open(CACHE_PATH, "rb") as f:
        full_en_df, full_en_occurrences, full_en_docs = pickle.load(f)
    print(f"{len(full_en_df):,} unique terms, {full_en_docs:,} docs, {full_en_occurrences:,} total token occurrences.")
else:
    full_en_df, full_en_occurrences, full_en_docs = parse_token_stream_dir(FULL_EN_TOKEN_STREAM)
    with open(CACHE_PATH, "wb") as f:
        pickle.dump((full_en_df, full_en_occurrences, full_en_docs), f)

In [ ]:
# Table 1: index composition stats

import pandas as pd

term_df_pd = pd.DataFrame({"token": list(full_en_df.keys()), "df": list(full_en_df.values())})
con.sql("CREATE OR REPLACE TEMP TABLE term_df AS SELECT * FROM term_df_pd")

unique_terms, total_postings, total_blocks = con.sql("""
    SELECT COUNT(*), SUM(df), SUM(CEIL(df / 128.0))::BIGINT
    FROM term_df
""").fetchone()

mean_doc_len = full_en_occurrences / full_en_docs

print(f"Unique terms:              {unique_terms:,}")
print(f"Total postings:            {total_postings:,}")
print(f"Total blocks (128/block):  {total_blocks:,}")
print(f"Mean document length:      {mean_doc_len:.2f}")

In [ ]:
# Table 2: document-frequency percentiles
#
# PERCENTILE_DISC (nearest observed df value), not PERCENTILE_CONT
# (would interpolate a fractional df, meaningless for a count) - matches
# the rank-based slicing query_gen.py already uses for its query sets.

p_min, p50, p90, p99, p9999, p_max = con.sql("""
    SELECT
        MIN(df),
        PERCENTILE_DISC(0.50)   WITHIN GROUP (ORDER BY df),
        PERCENTILE_DISC(0.90)   WITHIN GROUP (ORDER BY df),
        PERCENTILE_DISC(0.99)   WITHIN GROUP (ORDER BY df),
        PERCENTILE_DISC(0.9999) WITHIN GROUP (ORDER BY df),
        MAX(df)
    FROM term_df
""").fetchone()

for label, value in [("min", p_min), ("p50", p50), ("p90", p90), ("p99", p99), ("p99.99", p9999), ("max", p_max)]:
    print(f"{label:>7}: {value:,}")

In [ ]:
# Table 3: strata
#
# Same rank-based banding as query_gen.py's _percentile_slice (k = int(n *
# fraction), slice from the front for high-df, from the back for low-df),
# reimplemented in SQL so band membership matches how the benchmark query
# sets themselves were built. FLOOR(...)::BIGINT, not CAST(...AS BIGINT) -
# DuckDB's CAST rounds to nearest, but Python's int() (what query_gen.py
# actually uses) truncates toward zero; FLOOR matches int() exactly for
# positive values.

strata = con.sql("""
    WITH n_total AS (SELECT COUNT(*) AS n FROM term_df),
    ranked AS (
        SELECT df,
               ROW_NUMBER() OVER (ORDER BY df ASC)  AS asc_rank,
               ROW_NUMBER() OVER (ORDER BY df DESC) AS desc_rank
        FROM term_df
    )
    SELECT 'Rare (bottom 10%)' AS stratum, MIN(df) AS df_min, MAX(df) AS df_max, COUNT(*) AS n_terms
    FROM ranked, n_total
    WHERE asc_rank <= FLOOR(n_total.n * 0.10)::BIGINT
    UNION ALL
    SELECT 'Common (top 1%)', MIN(df), MAX(df), COUNT(*)
    FROM ranked, n_total
    WHERE desc_rank <= FLOOR(n_total.n * 0.01)::BIGINT
    UNION ALL
    SELECT 'Very common (top 0.01%)', MIN(df), MAX(df), COUNT(*)
    FROM ranked, n_total
    WHERE desc_rank <= FLOOR(n_total.n * 0.0001)::BIGINT
""").fetchall()

for stratum, df_min, df_max, n_terms in strata:
    print(f"{stratum:<26} df=[{df_min:,}, {df_max:,}]  terms={n_terms:,}")